## Preprocess the merged data

In [130]:
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [131]:
# import seaborn as sns
# import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np

In [132]:
# Load CSV files
data_dir = '../../mcphases/'

merged = pd.read_csv(os.path.join(data_dir, 'merged/physical_activity_merged.csv'))

print("CSV file loaded successfully!")

CSV file loaded successfully!


### 1. Examine missing values

In [133]:
# Number of missing values per column
missing_count = merged.isnull().sum()

# Percentage of missing values per column
missing_percent = ((merged.isnull().sum() / len(merged)) * 100).round(2)

# Combine into a table
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percent": missing_percent
}).sort_values("Missing Percent", ascending=False)

print(missing_summary)

                               Missing Count  Missing Percent
pdg                                     3795            67.06
daily_glucose                           2551            45.08
exerciselevel_num                       2329            41.16
fatigue_num                             2328            41.14
sedentary_activity                      1961            34.65
daily_hrv                                821            14.51
sleep_score                              587            10.37
sexually_active_num                      372             6.57
estrogen                                 321             5.67
lh                                       320             5.65
filtered_demographic_vo2_max             285             5.04
peak_zone                                209             3.69
cardio_zone                              209             3.69
below_fat_burn_zone                      209             3.69
fat_burn_zone                            209             3.69
moderate

In [134]:
merged.shape

(5659, 27)

42 participants * 2 periods * 90 days each period = 7560 rows max

In [135]:
merged.duplicated(subset=['id','day_in_study']).sum()

np.int64(0)

### 2. Process missing values

#### Unused
First, fill in the `study_interval` for time-series interpolation.

In [136]:
merged.groupby(['id', pd.cut(merged['day_in_study'], bins=[0,100,800,1004])])['study_interval'].unique()


C:\Users\caowe\AppData\Local\Temp\ipykernel_16420\1256506962.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  merged.groupby(['id', pd.cut(merged['day_in_study'], bins=[0,100,800,1004])])['study_interval'].unique()


id  day_in_study
1   (0, 100]        [2022]
    (100, 800]         NaN
    (800, 1004]        NaN
2   (0, 100]        [2022]
    (100, 800]         NaN
                     ...  
49  (100, 800]         NaN
    (800, 1004]        NaN
50  (0, 100]        [2022]
    (100, 800]         NaN
    (800, 1004]     [2024]
Name: study_interval, Length: 126, dtype: object

In [137]:
def day_range(day):
    if 1 <= day <= 100:
        return 'range_1'
    elif 800 <= day <= 1010:
        return 'range_2'
    return None

merged['_day_range'] = merged['day_in_study'].apply(day_range)

merged['study_interval'] = (
    merged.groupby(['id', '_day_range'])['study_interval']
    .transform(lambda x: x.ffill().bfill())
)

merged = merged.drop(columns='_day_range')

In [138]:
merged.isnull().sum()

id                                  0
study_interval                      0
is_weekend                          0
day_in_study                        0
sedentary_activity               1961
lightly_activity                  178
moderately_activity               178
very_activity                     178
calories_sum                        4
filtered_demographic_vo2_max      285
peak_zone                         209
cardio_zone                       209
fat_burn_zone                     209
below_fat_burn_zone               209
phase                               1
lh                                320
estrogen                          321
pdg                              3795
exerciselevel_num                2329
fatigue_num                      2328
daily_glucose                    2551
daily_hrv                         821
sleep_score                       587
age_of_first_menarche               0
age                                 0
menstrual_health_literacy_num      90
sexually_act

Now, we can process other features.

#### Used

In [139]:
# 1. Exclude features
merged = merged.drop(["sedentary_activity"], axis=1)

In [140]:
# 2. Deal with the flipped time in heart rate zone features: 'peak_zone', 'cardio_zone', 'fat_burn_zone', 'below_fat_burn_zone'
# Study interval 2 reverses study interval 1's column orders
cols = ['peak_zone', 'cardio_zone', 'fat_burn_zone', 'below_fat_burn_zone']
reversed_cols = cols[::-1]  # ['below_fat_burn_zone', 'fat_burn_zone', 'cardio_zone', 'peak_zone']

mask = merged['study_interval'] == 2024.0

merged.loc[mask, cols] = merged.loc[mask, reversed_cols].values

In [141]:
merged[cols]

,peak_zone,cardio_zone,fat_burn_zone,below_fat_burn_zone
0,0.0,0.0,126.0,1036.0
1,5.0,82.0,416.0,512.0
2,5.0,119.0,599.0,368.0
3,0.0,0.0,212.0,613.0
4,8.0,123.0,250.0,308.0
...,...,...,...,...
5654,0.0,0.0,39.0,1285.0
5655,0.0,9.0,65.0,1241.0
5656,0.0,0.0,6.0,1410.0
5657,0.0,0.0,24.0,1409.0


In [142]:
# 3. Impute missing values — participant-level median (pre-split safe)
PARTICIPANT_MEDIAN_COLS = [
    "lightly_activity", "moderately_activity", "very_activity", "calories_sum", "peak_zone", "cardio_zone",
    "fat_burn_zone", "below_fat_burn_zone", "exerciselevel_num",
    "daily_glucose", "daily_hrv", "sleep_score",
]
for col in PARTICIPANT_MEDIAN_COLS:
    merged[col] = merged.groupby("id", sort=False)[col].transform(
        lambda s: s.fillna(s.median())
    )

d:\ML-for-Women\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\ML-for-Women\.venv\lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [143]:
merged.isnull().sum()

id                                  0
study_interval                      0
is_weekend                          0
day_in_study                        0
lightly_activity                    0
moderately_activity                 0
very_activity                       0
calories_sum                        0
filtered_demographic_vo2_max      285
peak_zone                           0
cardio_zone                         0
fat_burn_zone                       0
below_fat_burn_zone                 0
phase                               1
lh                                320
estrogen                          321
pdg                              3795
exerciselevel_num                   0
fatigue_num                      2328
daily_glucose                       0
daily_hrv                         180
sleep_score                         0
age_of_first_menarche               0
age                                 0
menstrual_health_literacy_num      90
sexually_active_num               372
dtype: int64

In [144]:
# Drop rows with no recoverable daily_hrv after participant median impute
n_before = len(merged)
merged = merged.dropna(subset=["daily_hrv"])
print(f"Dropped {n_before - len(merged):,} rows missing daily_hrv ({n_before:,} -> {len(merged):,})")


Dropped 180 rows missing daily_hrv (5,659 -> 5,479)


`menstrual_health_literacy_num` should be handled AFTER splitting out the test set -- will perform a global median imputation.

However, in the previous code cell, dropping rows with missing daily_hrv also drops all rows with missing menstrual_health_literacy_num.

In [145]:
# Check for all-NaN participants after participant-level median imputation
remaining = (
    merged.groupby("id")[PARTICIPANT_MEDIAN_COLS]
    .apply(lambda g: g.isna().all())
    .stack()
    .reset_index()
    .rename(columns={"level_1": "column", 0: "all_nan"})
)
all_nan_cases = remaining[remaining["all_nan"]]
if all_nan_cases.empty:
    print("No all-NaN participant/column pairs after participant-level median.")
else:
    print(f"Found {len(all_nan_cases)} all-NaN participant/column pairs:")
    display(all_nan_cases)

No all-NaN participant/column pairs after participant-level median.


In [146]:
merged.isnull().sum()

id                                  0
study_interval                      0
is_weekend                          0
day_in_study                        0
lightly_activity                    0
moderately_activity                 0
very_activity                       0
calories_sum                        0
filtered_demographic_vo2_max      276
peak_zone                           0
cardio_zone                         0
fat_burn_zone                       0
below_fat_burn_zone                 0
phase                               1
lh                                308
estrogen                          309
pdg                              3615
exerciselevel_num                   0
fatigue_num                      2301
daily_glucose                       0
daily_hrv                           0
sleep_score                         0
age_of_first_menarche               0
age                                 0
menstrual_health_literacy_num       0
sexually_active_num               372
dtype: int64

In [147]:
# 4. Interpolate missing values for specific features
# First check that there are no duplicate day_in_study values for each id and study_interval combination
merged.groupby(['id', 'study_interval'])['day_in_study'].apply(lambda x: x.duplicated().any()).any()

np.False_

In [148]:
# Now we can safely interpolate the missing values for the specified features
features = ['filtered_demographic_vo2_max', 'lh', 'estrogen']

merged = merged.sort_values(['id', 'study_interval', 'day_in_study'])

merged = merged.set_index('day_in_study')
merged[features] = (
    merged.groupby(['id', 'study_interval'])[features]
    .transform(lambda x: x.interpolate(method='index', limit_direction='both'))
)
merged = merged.reset_index()

`phase` has only 1 missing vlue, let's take a look!

In [149]:
# Find the row
merged[merged['phase'].isna()]

,day_in_study,id,study_interval,is_weekend,lightly_activity,moderately_activity,very_activity,calories_sum,filtered_demographic_vo2_max,peak_zone,...,pdg,exerciselevel_num,fatigue_num,daily_glucose,daily_hrv,sleep_score,age_of_first_menarche,age,menstrual_health_literacy_num,sexually_active_num
95,6,4,2022,False,234.0,34.0,20.0,2288.0,32.57471,0.0,...,NaN,2.0,0.0,6.0,80.3655,77.0,12,24,2.0,0.0


In [150]:
missing_row = merged[merged['phase'].isna()]
id_val = missing_row['id'].values[0]
interval_val = missing_row['study_interval'].values[0]

merged[(merged['id'] == id_val) & (merged['study_interval'] == interval_val)][['day_in_study', 'phase']].sort_values('day_in_study').head(20)

,day_in_study,phase
90,1,Menstrual
91,2,Follicular
92,3,Follicular
93,4,Follicular
94,5,Fertility
95,6,NaN
96,7,Fertility
97,8,Fertility
98,9,Fertility
99,10,Fertility


Obviously, the nan value should be `'Fertility'`.

In [151]:
# row 95
merged.loc[95, 'phase'] = 'Fertility'

In [152]:
# 5. Fill in a special value for the feature 'sexually_active_num'
merged['sexually_active_num'] = merged['sexually_active_num'].fillna(-1)

___

In [153]:
# Number of missing values per column
missing_count = merged.isnull().sum()

# Percentage of missing values per column
missing_percent = ((merged.isnull().sum() / len(merged)) * 100).round(2)

# Combine into a table
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percent": missing_percent
}).sort_values("Missing Percent", ascending=False)

print(missing_summary)

                               Missing Count  Missing Percent
pdg                                     3615            65.98
fatigue_num                             2301            42.00
study_interval                             0             0.00
is_weekend                                 0             0.00
lightly_activity                           0             0.00
moderately_activity                        0             0.00
very_activity                              0             0.00
calories_sum                               0             0.00
day_in_study                               0             0.00
id                                         0             0.00
peak_zone                                  0             0.00
filtered_demographic_vo2_max               0             0.00
below_fat_burn_zone                        0             0.00
cardio_zone                                0             0.00
phase                                      0             0.00
lh      

Now, let's handle the 2/3 missingness of `pdg`!

In [154]:
# Is missingness random, or structured?
merged.groupby('study_interval')['pdg'].apply(lambda x: x.isna().mean())

study_interval
2022    1.000000
2024    0.049465
Name: pdg, dtype: float64

`pdg` simply wasn't collected at all during the first study interval, likely because it wasn't part of the study protocol yet, or that assay was added later.

Since `pdg` is one of the hormones used to predict the phase label for the Mira device, the `phase` feature include some information about the pdg hormone, to a certian degree. Thus, it's reasonable to preclude this feature.

In [155]:
merged = merged.drop(["pdg"], axis=1)

___

Now, let's handle the missing values in the target variable!

In [156]:
merged_fatigue = (
    merged[merged['fatigue_num'].notna()]
    .copy()
)

In [157]:
merged_fatigue.shape

(3178, 25)

In [158]:
# Number of missing values per column
missing_count = merged_fatigue.isnull().sum()

# Percentage of missing values per column
missing_percent = ((merged_fatigue.isnull().sum() / len(merged_fatigue)) * 100).round(2)

# Combine into a table
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percent": missing_percent
}).sort_values("Missing Percent", ascending=False)

print(missing_summary)

                               Missing Count  Missing Percent
day_in_study                               0              0.0
id                                         0              0.0
study_interval                             0              0.0
is_weekend                                 0              0.0
lightly_activity                           0              0.0
moderately_activity                        0              0.0
very_activity                              0              0.0
calories_sum                               0              0.0
filtered_demographic_vo2_max               0              0.0
peak_zone                                  0              0.0
cardio_zone                                0              0.0
fat_burn_zone                              0              0.0
below_fat_burn_zone                        0              0.0
phase                                      0              0.0
lh                                         0              0.0
estrogen

### 3. Signal smoothing


Slow-varying signals only: `lh`, `estrogen`. Compare multiple smoothers (causal and centered) before choosing whether to apply any.


Always smooth within `id` and `study_interval`, not globally.


In [159]:
# makes sure the data is sorted by id, study_interval, and day_in_study before applying smoothing functions (based on time)
merged_fatigue = merged_fatigue.sort_values(['id', 'study_interval', 'day_in_study'])

smooth_features = [
    'lh',
    'estrogen'
]

# ewm: exponentially weighted moving average, which gives more weight to recent observations.
# The 'window' parameter controls the degree of weighting decrease, with a smaller window giving more weight to recent observations.
SMOOTH_CONFIGS = [
    {'method': 'roll_mean_center', 'window': 3},
    {'method': 'roll_mean_center', 'window': 7},
    {'method': 'roll_mean_causal', 'window': 3},
    {'method': 'roll_mean_causal', 'window': 7},
    {'method': 'roll_median_center', 'window': 3},
    {'method': 'roll_median_center', 'window': 7},
    {'method': 'roll_median_causal', 'window': 3},
    {'method': 'roll_median_causal', 'window': 7},
    {'method': 'ewm', 'window': 3},
    {'method': 'ewm', 'window': 7}
]

GROUP_COLS = ['id', 'study_interval']


In [160]:
def apply_smoothing(series, method, window):
    if method == 'roll_mean_center':
        return series.rolling(window, center=True, min_periods=1).mean()
    if method == 'roll_mean_causal':
        return series.rolling(window, min_periods=1).mean()
    if method == 'roll_median_center':
        return series.rolling(window, center=True, min_periods=1).median()
    if method == 'roll_median_causal':
        return series.rolling(window, min_periods=1).median()
    if method == 'ewm':
        return series.ewm(span=window, adjust=False, min_periods=1).mean()
    raise ValueError(f'Unknown method: {method}')


def count_iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()

# noise metric
def mean_abs_step_grouped(df, col, group_cols):
    return (
        df.groupby(group_cols, sort=False)[col]
        .transform(lambda s: s.diff().abs())
        .mean()
    )


def smooth_column(df, col, method, window, group_cols):
    return (
        df.groupby(group_cols, sort=False)[col]
        .transform(lambda s: apply_smoothing(s, method, window))
    )


def select_best_smoother(comparison, feature, corr_min=0.85):
    """Return (method, window) for highest noise_reduction_pct with corr >= corr_min, or None."""
    sub = comparison[
        (comparison['feature'] == feature) & (comparison['corr_with_raw'] >= corr_min)
    ]
    if sub.empty:
        return None
    best = sub.sort_values('noise_reduction_pct', ascending=False).iloc[0]
    return best['method'], int(best['window'])


def compute_smoothing_comparison(df, features, configs, group_cols):
    rows = []
    for feat in features:
        raw = df[feat]
        raw_std = raw.std()
        raw_step = mean_abs_step_grouped(df, feat, group_cols)
        raw_skew = raw.skew()
        raw_outliers = count_iqr_outliers(raw)

        for cfg in configs:
            method, window = cfg['method'], cfg['window']
            smooth = smooth_column(df, feat, method, window, group_cols)
            smooth_std = smooth.std()
            smooth_step = mean_abs_step_grouped(
                df.assign(_tmp=smooth), '_tmp', group_cols
            )
            std_red = (1 - smooth_std / raw_std) * 100 if raw_std else 0.0
            noise_red = (1 - smooth_step / raw_step) * 100 if raw_step else 0.0

            rows.append({
                'feature': feat,
                'method': method,
                'window': window,
                'std_raw': round(raw_std, 4),
                'std_smooth': round(smooth_std, 4),
                'std_reduction_pct': round(std_red, 2),
                'mean_abs_step_raw': round(raw_step, 4),
                'mean_abs_step_smooth': round(smooth_step, 4),
                'noise_reduction_pct': round(noise_red, 2),
                'corr_with_raw': round(raw.corr(smooth), 4),
                'skew_raw': round(raw_skew, 2),
                'skew_smooth': round(smooth.skew(), 2),
                'outliers_iqr_raw': int(raw_outliers),
                'outliers_iqr_smooth': int(count_iqr_outliers(smooth))
            })
    return pd.DataFrame(rows)


In [161]:
comparison = compute_smoothing_comparison(
    merged_fatigue, smooth_features, SMOOTH_CONFIGS, GROUP_COLS
)

# noise_reduction_pct: primary metric — higher means day-to-day jumps were reduced more.
display(
    comparison.sort_values(['feature', 'noise_reduction_pct'], ascending=[True, False])
)


,feature,method,window,std_raw,std_smooth,std_reduction_pct,mean_abs_step_raw,mean_abs_step_smooth,noise_reduction_pct,corr_with_raw,skew_raw,skew_smooth,outliers_iqr_raw,outliers_iqr_smooth
15,estrogen,roll_median_center,7,111.2554,81.3593,26.87,63.2052,11.5778,81.68,0.7784,2.14,2.27,192,123
17,estrogen,roll_median_causal,7,111.2554,84.2033,24.32,63.2052,12.1912,80.71,0.6615,2.14,2.36,192,130
11,estrogen,roll_mean_center,7,111.2554,82.8906,25.50,63.2052,12.2518,80.62,0.7920,2.14,1.63,192,106
13,estrogen,roll_mean_causal,7,111.2554,85.6640,23.00,63.2052,12.8568,79.66,0.7015,2.14,1.75,192,116
19,estrogen,ewm,7,111.2554,84.1217,24.39,63.2052,14.9444,76.36,0.7973,2.14,1.77,192,107
14,estrogen,roll_median_center,3,111.2554,95.5555,14.11,63.2052,24.5696,61.13,0.8541,2.14,2.23,192,181
16,estrogen,roll_median_causal,3,111.2554,96.5951,13.18,63.2052,24.7674,60.81,0.7908,2.14,2.25,192,183
10,estrogen,roll_mean_center,3,111.2554,94.0023,15.51,63.2052,25.1906,60.14,0.8654,2.14,1.85,192,176
12,estrogen,roll_mean_causal,3,111.2554,95.0402,14.57,63.2052,25.4068,59.80,0.8369,2.14,1.88,192,179
18,estrogen,ewm,3,111.2554,93.2644,16.17,63.2052,28.9938,54.13,0.9217,2.14,1.88,192,154


In [162]:
# Need to preserve at least 85% information
CORR_MIN = 0.85
BEST_SMOOTH = {}

for feat in smooth_features:
    picked = select_best_smoother(comparison, feat, CORR_MIN)
    if picked is None:
        print(
            f"\n=== {feat}: no smoothing method keeps correlation with raw >= {CORR_MIN} "
            f"with raw — skip smoothing for this feature ==="
        )
        BEST_SMOOTH[feat] = None
        continue

    BEST_SMOOTH[feat] = {'method': picked[0], 'window': picked[1]}

    sub = comparison[
        (comparison['feature'] == feat) & (comparison['corr_with_raw'] >= CORR_MIN)
    ]
    best_row = sub.sort_values('noise_reduction_pct', ascending=False).iloc[[0]]

    print(f"\n=== {feat}: best method (corr >= {CORR_MIN}) ===")
    display(best_row)

    best = best_row.iloc[0]
    raw = merged_fatigue[feat]
    smooth = smooth_column(
        merged_fatigue, feat, best['method'], int(best['window']), GROUP_COLS
    )
    snapshot = pd.DataFrame({
        'raw': [
            raw.mean(), raw.std(), raw.skew(),
            mean_abs_step_grouped(merged_fatigue, feat, GROUP_COLS),
        ],
        'best_smooth': [
            smooth.mean(), smooth.std(), smooth.skew(),
            mean_abs_step_grouped(
                merged_fatigue.assign(_tmp=smooth), '_tmp', GROUP_COLS
            ),
        ],
    }, index=['mean', 'std', 'skew', 'mean_abs_step'])
    method_label = f"{best['method']}_w{int(best['window'])}"
    print(f"Raw vs best ({method_label}):")
    display(snapshot)



=== lh: best method (corr >= 0.85) ===


,feature,method,window,std_raw,std_smooth,std_reduction_pct,mean_abs_step_raw,mean_abs_step_smooth,noise_reduction_pct,corr_with_raw,skew_raw,skew_smooth,outliers_iqr_raw,outliers_iqr_smooth
8,lh,ewm,3,6.8603,4.8622,29.13,3.2156,1.5839,50.74,0.8801,5.55,3.73,274,288


Raw vs best (ewm_w3):


,raw,best_smooth
mean,5.345806,5.353859
std,6.860294,4.862155
skew,5.545970,3.727780
mean_abs_step,3.215587,1.583863



=== estrogen: best method (corr >= 0.85) ===


,feature,method,window,std_raw,std_smooth,std_reduction_pct,mean_abs_step_raw,mean_abs_step_smooth,noise_reduction_pct,corr_with_raw,skew_raw,skew_smooth,outliers_iqr_raw,outliers_iqr_smooth
14,estrogen,roll_median_center,3,111.2554,95.5555,14.11,63.2052,24.5696,61.13,0.8541,2.14,2.23,192,181


Raw vs best (roll_median_center_w3):


,raw,best_smooth
mean,137.777237,131.355745
std,111.255401,95.555496
skew,2.141893,2.225838
mean_abs_step,63.205155,24.569638


In [163]:
applied = []

for feat in smooth_features:
    cfg = BEST_SMOOTH.get(feat)

    col_smooth = f"{feat}_smooth"
    merged_fatigue[col_smooth] = smooth_column(
        merged_fatigue, feat, cfg['method'], cfg['window'], GROUP_COLS
    )
    applied.append({
        'feature': feat,
        'smooth_column': col_smooth,
        'method': cfg['method'],
        'window': cfg['window'],
    })

print(f"Applied smoothing to {len(applied)} feature(s).")
display(pd.DataFrame(applied))


Applied smoothing to 2 feature(s).


,feature,smooth_column,method,window
0,lh,lh_smooth,ewm,3
1,estrogen,estrogen_smooth,roll_median_center,3


### 4. Look at the distribution of the features

In [164]:
merged_fatigue.columns

Index(['day_in_study', 'id', 'study_interval', 'is_weekend',
       'lightly_activity', 'moderately_activity', 'very_activity',
       'calories_sum', 'filtered_demographic_vo2_max', 'peak_zone',
       'cardio_zone', 'fat_burn_zone', 'below_fat_burn_zone', 'phase', 'lh',
       'estrogen', 'exerciselevel_num', 'fatigue_num', 'daily_glucose',
       'daily_hrv', 'sleep_score', 'age_of_first_menarche', 'age',
       'menstrual_health_literacy_num', 'sexually_active_num', 'lh_smooth',
       'estrogen_smooth'],
      dtype='object')

In [165]:
merged_fatigue = merged_fatigue.drop(columns=["lh", "estrogen"])
merged_fatigue

,day_in_study,id,study_interval,is_weekend,lightly_activity,moderately_activity,very_activity,calories_sum,filtered_demographic_vo2_max,peak_zone,...,fatigue_num,daily_glucose,daily_hrv,sleep_score,age_of_first_menarche,age,menstrual_health_literacy_num,sexually_active_num,lh_smooth,estrogen_smooth
0,1,2,2022,True,89.0,0.0,0.0,1568.00,39.95957,0.0,...,2.0,8.40,43.9695,87.0,13,29,3.0,1.0,2.900000,102.70
1,2,2,2022,False,82.0,0.0,0.0,1558.00,40.27132,0.0,...,3.0,6.50,43.9695,90.0,13,29,3.0,1.0,3.050000,126.90
2,3,2,2022,False,86.0,0.0,0.0,1579.00,40.48366,0.0,...,2.0,6.30,43.9695,89.0,13,29,3.0,1.0,4.375000,187.50
3,4,2,2022,False,143.0,26.0,31.0,1958.00,40.61939,0.0,...,4.0,6.30,43.9695,80.0,13,29,3.0,1.0,15.387500,220.30
4,5,2,2022,False,115.0,17.0,22.0,1804.00,40.61293,0.0,...,4.0,6.40,43.9695,71.0,13,29,3.0,1.0,17.443750,220.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5371,86,50,2022,False,196.0,23.0,33.0,2107.00,43.28141,0.0,...,5.0,6.30,47.9555,56.0,11,20,2.0,0.0,3.628999,54.60
5372,87,50,2022,False,282.0,13.0,7.0,2131.00,43.36550,0.0,...,3.0,6.30,54.4515,84.0,11,20,2.0,0.0,3.664499,28.10
5373,88,50,2022,False,181.0,0.0,0.0,1759.00,43.47867,0.0,...,3.0,6.70,57.6890,69.0,11,20,2.0,0.0,2.982250,22.10
5374,89,50,2022,False,193.0,30.0,44.0,2211.00,43.60040,0.0,...,3.0,6.95,62.2390,74.0,11,20,2.0,0.0,2.841125,20.85


Keep `study_interval` in the exported CSV as metadata for wave-aware temporal features in modeling. It is **not** used as a model input feature.

In [166]:
merged_fatigue.columns

Index(['day_in_study', 'id', 'study_interval', 'is_weekend',
       'lightly_activity', 'moderately_activity', 'very_activity',
       'calories_sum', 'filtered_demographic_vo2_max', 'peak_zone',
       'cardio_zone', 'fat_burn_zone', 'below_fat_burn_zone', 'phase',
       'exerciselevel_num', 'fatigue_num', 'daily_glucose', 'daily_hrv',
       'sleep_score', 'age_of_first_menarche', 'age',
       'menstrual_health_literacy_num', 'sexually_active_num', 'lh_smooth',
       'estrogen_smooth'],
      dtype='object')

#### Categorical features

'is_weekend', 'phase', 'exerciselevel_num', 'fatigue_num', 'age_of_first_menarche', 'age', 'menstrual_health_literacy_num', 'sexually_active_num'

In [167]:
categorical_features_by_day = ['is_weekend', 'phase', 'exerciselevel_num', 'fatigue_num']
categorical_features_by_participant = ['menstrual_health_literacy_num', 'sexually_active_num']

In [168]:
# Day-level: counts and proportion
phase_order = ['Menstrual', 'Follicular', 'Fertility', 'Luteal']

for col in categorical_features_by_day:
    print(f"\n=== {col} ===")

    if col == 'phase':
        counts = merged_fatigue[col].value_counts(dropna=False).reindex(phase_order)
        props = merged_fatigue[col].value_counts(normalize=True, dropna=False).reindex(phase_order)
    else:
        counts = merged_fatigue[col].value_counts(dropna=False).sort_index()
        props = merged_fatigue[col].value_counts(normalize=True, dropna=False).sort_index()

    display(pd.DataFrame({'count': counts, 'proportion': props.round(3)}))


# Participant-level: one row per id
participants = merged_fatigue.drop_duplicates(subset='id')

for col in categorical_features_by_participant:
    print(f"\n=== {col} (per participant) ===")
    counts = participants[col].value_counts(dropna=False).sort_index()
    props = participants[col].value_counts(normalize=True, dropna=False).sort_index()
    display(pd.DataFrame({'count': counts, 'proportion': props.round(3)}))


=== is_weekend ===


,count,proportion
is_weekend,,
False,2280,0.717
True,898,0.283



=== phase ===


,count,proportion
phase,,
Menstrual,607,0.191
Follicular,814,0.256
Fertility,700,0.220
Luteal,1057,0.333



=== exerciselevel_num ===


...

,count,proportion
fatigue_num,,
0.0,386,0.121
1.0,407,0.128
2.0,549,0.173
3.0,928,0.292
4.0,669,0.211
5.0,239,0.075



=== menstrual_health_literacy_num (per participant) ===


,count,proportion
menstrual_health_literacy_num,,
0.0,1,0.025
1.0,4,0.100
2.0,23,0.575
3.0,11,0.275
4.0,1,0.025



=== sexually_active_num (per participant) ===


,count,proportion
sexually_active_num,,
-1.0,3,0.075
0.0,23,0.575
1.0,14,0.350


Since only 6 datapoints (less than 1%) recorded exercise level of 0, it should be merged with similar, in this case, level 1, categories.

The same holds for menstrual_health_literacy_num. There are only 42 participants, so only 1 participant in a category is not representative of the whole distribution. (NaN will be handeled after splitting to avoid data leakage)

In [169]:
merged_fatigue['exerciselevel_num'] = merged_fatigue['exerciselevel_num'].replace({0: 1})

In [170]:
merged_fatigue['menstrual_health_literacy_num'] = merged_fatigue['menstrual_health_literacy_num'].replace({0: 1, 4: 3})

#### Numerical features

'lightly_activity', 'moderately_activity', 'very_activity', 'calories_sum', 'filtered_demographic_vo2_max',
'filtered_demographic_vo2_max_error', 'peak_zone', 'cardio_zone', 'daily_glucose', 'daily_hrv',
'sleep_score', 'fat_burn_zone', 'below_fat_burn_zone', 'lh_smooth', 'estrogen_smooth'

In [171]:
numerical_features = [
    'lightly_activity', 'moderately_activity', 'very_activity', 'calories_sum',
    'filtered_demographic_vo2_max', "sleep_score", "daily_glucose", "daily_hrv", 
    'peak_zone', 'cardio_zone', 'fat_burn_zone', 'below_fat_burn_zone',
    'lh_smooth', 'estrogen_smooth', 'age_of_first_menarche', 'age'
]

In [172]:
display(merged_fatigue[numerical_features].describe().T)

,count,mean,std,min,25%,50%,75%,max
lightly_activity,3178.0,168.901982,99.068502,0.00000,102.000000,167.000000,228.000000,572.000000
moderately_activity,3178.0,16.067810,23.861489,0.00000,0.000000,7.000000,24.000000,236.000000
very_activity,3178.0,17.272184,27.144895,0.00000,0.000000,5.000000,25.000000,264.000000
calories_sum,3178.0,1989.257772,526.347628,1440.00000,1711.000000,1892.000000,2153.750000,20301.150000
filtered_demographic_vo2_max,3178.0,38.642582,4.887862,25.43059,35.437683,38.985650,41.639960,52.518780
sleep_score,3178.0,76.968534,8.309813,37.00000,72.000000,78.000000,83.000000,94.000000
daily_glucose,3178.0,10.812358,22.212415,2.90000,5.600000,6.000000,6.500000,157.500000
daily_hrv,3178.0,52.332552,24.782667,0.00000,33.805250,47.446750,66.158500,186.615000
peak_zone,3178.0,0.404972,3.628213,0.00000,0.000000,0.000000,0.000000,121.000000
cardio_zone,3178.0,7.791378,17.967854,0.00000,0.000000,1.000000,8.000000,392.000000


In [173]:
rows = []
for col in numerical_features:
    s = merged_fatigue[col]
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    rows.append({
        'feature': col,
        'skew': s.skew().round(2),
        'pct_zero': (s == 0).mean().round(3),
        'outliers_iqr': ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()
})
display(pd.DataFrame(rows).set_index('feature'))

,skew,pct_zero,outliers_iqr
feature,,,
lightly_activity,0.30,0.087,33
moderately_activity,2.68,0.399,165
very_activity,2.68,0.421,215
calories_sum,14.35,0.000,120
filtered_demographic_vo2_max,0.06,0.000,82
sleep_score,-0.88,0.000,58
daily_glucose,4.46,0.000,199
daily_hrv,1.06,0.001,67
peak_zone,20.30,0.949,162


In [174]:
# Upper 3-sigma cap: report count, then clip (measurement features only)
CAP_GROUPS = {
    "activity": ["lightly_activity", "moderately_activity", "very_activity"],
    "calories": ["calories_sum"],
    "zones": ["cardio_zone", "fat_burn_zone", "below_fat_burn_zone"],
    "glucose": ["daily_glucose"],
    "hrv": ["daily_hrv"],
    "sleep": ["sleep_score"],
    "vo2": ["filtered_demographic_vo2_max"],
}

for group, cols in CAP_GROUPS.items():
    print(f"\n=== {group} ===")
    for col in cols:
        s = merged_fatigue[col]
        upper = s.mean() + 3 * s.std()
        n_cap = (s > upper).sum()
        print(f"  {col}: capping {n_cap} values above {upper:.2f}")
        merged_fatigue[col] = s.clip(upper=upper)


=== activity ===
  lightly_activity: capping 10 values above 466.11
  moderately_activity: capping 71 values above 87.65
  very_activity: capping 67 values above 98.71

=== calories ===
  calories_sum: capping 27 values above 3568.30

=== zones ===
  cardio_zone: capping 56 values above 61.69
  fat_burn_zone: capping 47 values above 638.33
  below_fat_burn_zone: capping 0 values above 1806.79

=== glucose ===
  daily_glucose: capping 142 values above 77.45

=== hrv ===
  daily_hrv: capping 37 values above 126.68

=== sleep ===
  sleep_score: capping 0 values above 101.90

=== vo2 ===
  filtered_demographic_vo2_max: capping 0 values above 53.31


In [175]:
# "peak_zone": 94% of values are zero, meaning significant "class imbalance", so dropping it would be wise.
merged_fatigue = merged_fatigue.drop(columns=['peak_zone'])

In [176]:
# Zero-inflated / skewed activity features: log1p transform
for col in ['lightly_activity', 'moderately_activity', 'very_activity', 'cardio_zone']:
    merged_fatigue[col] = np.log1p(merged_fatigue[col])

### 5. Export

In [177]:
merged_fatigue.to_csv("../../mcphases/merged/physical_activity_merged_processed.csv", index=False)